In [22]:
team_matches.to_csv("../data/features/match_features.csv", index=False)
team_matches.shape

(760, 24)

In [21]:
import os
os.makedirs("../data/features", exist_ok=True)

In [19]:
team_matches = team_matches.merge(
    h2h_df[["team", "utcDate", "h2h_win_pct"]],
    on=["team", "utcDate"],
    how="left"
)

In [16]:
h2h_df["h2h_win"] = (h2h_df["goals_for"] > h2h_df["goals_against"]).astype(int)

h2h_df["h2h_win_pct"] = (
    h2h_df.groupby(["team", "opponent"])["h2h_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [15]:
h2h_records = []

for _, match in clean_matches.iterrows():
    h2h_records.append({"team": match["home_team_name"], "opponent": match["away_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["home_goals"], "goals_against": match["away_goals"]})
    h2h_records.append({"team": match["away_team_name"], "opponent": match["home_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["away_goals"], "goals_against": match["home_goals"]})

h2h_df = pd.DataFrame(h2h_records).sort_values(["team", "opponent", "utcDate"]).reset_index(drop=True)

In [13]:
team_matches["is_win"] = (team_matches["result"] == "WIN").astype(int)

team_matches["home_win_pct"] = (
    team_matches[team_matches["is_home"]]
    .groupby("team")["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["away_win_pct"] = (
    team_matches[~team_matches["is_home"]]
    .groupby("team")["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [12]:
team_matches["days_since_last_match"] = (
    team_matches.groupby("team")["utcDate"]
    .diff()
    .dt.days
)

In [11]:
team_matches["clean_sheet"] = (team_matches["goals_conceded"] == 0).astype(int)
team_matches["clean_sheet_pct"] = (
    team_matches.groupby("team")["clean_sheet"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [10]:
team_matches["goal_difference"] = team_matches["goals_scored"] - team_matches["goals_conceded"]

In [8]:
team_matches["avg_goals_scored"] = (
    team_matches.groupby("team")["goals_scored"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["avg_goals_conceded"] = (
    team_matches.groupby("team")["goals_conceded"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [7]:
def compute_streak(results, target):
    streak = 0
    streaks = []
    for r in results:
        streaks.append(streak)  # streak BEFORE this match
        streak = streak + 1 if r == target else 0
    return streaks

team_matches["win_streak"] = team_matches.groupby("team")["result"].transform(
    lambda x: compute_streak(x, "WIN")
)
team_matches["loss_streak"] = team_matches.groupby("team")["result"].transform(
    lambda x: compute_streak(x, "LOSS")
)

In [6]:
team_matches["rolling_form_pts"] = (
    team_matches.groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

In [5]:
points_map = {"WIN": 3, "DRAW": 1, "LOSS": 0}
team_matches["points"] = team_matches["result"].map(points_map)

In [2]:
home_rows = clean_matches[["utcDate", "matchday", "id", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
home_rows.columns = ["utcDate", "matchday", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
home_rows["is_home"] = True
home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

away_rows = clean_matches[["utcDate", "matchday", "id", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
away_rows.columns = ["utcDate", "matchday", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
away_rows["is_home"] = False
away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

team_matches = pd.concat([home_rows, away_rows]).sort_values(["team", "utcDate"]).reset_index(drop=True)
team_matches.drop(columns="winner", inplace=True)

In [1]:
import pandas as pd

clean_matches = pd.read_csv("../data/processed/clean_matches.csv", parse_dates=["utcDate"])
clean_matches = clean_matches.sort_values("utcDate").reset_index(drop=True)